# Pharmasense — ST-GNN: Spatial-Temporal Outbreak Prediction
**GraphSAGE + GRU on India's district adjacency graph. Predicts malaria + dengue + cholera.**

- Diseases: malaria, dengue, cholera (spread across district borders)
- Architecture: GraphSAGE (2-hop spatial) + GRU (24-week temporal)
- Downloads all data from GitHub — no DB needed

## Before running
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run All**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/01_train_stgnn.ipynb)

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f'GPU: {r.stdout.strip()}')
else:
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.2.0', '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch_geometric', 'torch_scatter', 'torch_sparse', 'torch_cluster',
    '-f', 'https://data.pyg.org/whl/torch-2.2.0+cu118.html'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pandas', 'numpy', 'scikit-learn', 'tqdm', 'matplotlib', 'seaborn'], check=True)
print('All dependencies installed.')

In [ ]:
GITHUB_RAW = 'https://raw.githubusercontent.com/abhi-dev99/agtechathon/main/ml/data'
DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
import os
from google.colab import files
REQUIRED_FILES = ['malaria_weekly.csv.gz', 'epiclim_clean.csv', 'district_index.json']
missing = [f for f in REQUIRED_FILES if not os.path.exists(f'/content/{f}')]
if missing:
    print('GitHub repository is private. Please upload the following files from your local ml/data/ folder:\n')
    for f in missing:
        print(f'  - {f}')
    print('\n')
    uploaded = files.upload()
    missing = [f for f in REQUIRED_FILES if not os.path.exists(f'/content/{f}')]
    if missing:
        raise FileNotFoundError(f'Still missing: {missing}')
print('All data files are present.')


In [ ]:
import json, pandas as pd, numpy as np
import torch
import torch.nn as nn
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Load graph structure ──────────────────────────────────────────────────────
with open('/content/district_index.json') as f:
    graph_data = json.load(f)

node_list   = graph_data['nodes']   # list of {id, name, state, lat, lon}
edge_list   = graph_data['edges']   # list of [src_idx, dst_idx]
N_NODES     = len(node_list)
name_to_idx = {n['name'].lower().strip(): i for i, n in enumerate(node_list)}

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous().to(DEVICE)
print(f'Graph: {N_NODES} nodes, {len(edge_list)} edges')

In [ ]:
# ── Load and align case data ──────────────────────────────────────────────────
mal = pd.read_csv('/content/malaria_weekly.csv.gz', parse_dates=['week_start_date'])
mal_weekly = mal.groupby(['week_start_date','district'])['cases'].sum().reset_index()
mal_weekly['disease'] = 'malaria'

epi = pd.read_csv('/content/epiclim_clean.csv')
epi['week_of_outbreak'] = pd.to_datetime(epi[['year','mon','day']].rename(columns={'mon':'month'}))
epi['Cases'] = pd.to_numeric(epi['Cases'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
dengue = epi[epi['Disease'].str.lower().str.contains('dengue', na=False)].copy()
dengue_weekly = dengue.groupby(['week_of_outbreak','district'])['Cases'].sum().reset_index()
dengue_weekly.columns = ['week_start_date','district','cases']
dengue_weekly['disease'] = 'dengue'

cholera = epi[epi['Disease'].str.lower().str.contains('cholera', na=False)].copy()
cholera_weekly = cholera.groupby(['week_of_outbreak','district'])['Cases'].sum().reset_index()
cholera_weekly.columns = ['week_start_date','district','cases']
cholera_weekly['disease'] = 'cholera'

cases = pd.concat([mal_weekly, dengue_weekly, cholera_weekly], ignore_index=True)
cases['district_lower'] = cases['district'].str.lower().str.strip()
cases['node_idx'] = cases['district_lower'].map(name_to_idx)
cases = cases.dropna(subset=['node_idx'])
cases['node_idx'] = cases['node_idx'].astype(int)

weeks = sorted(cases['week_start_date'].unique())
week_to_t = {w: i for i, w in enumerate(weeks)}
T = len(weeks)
print(f'Time steps: {T} | Matched districts: {cases.node_idx.nunique()}')

In [ ]:
# ── Build node feature matrix [T, N, D] ──────────────────────────────────────
D = 3  # diseases
disease_names = ['malaria','dengue','cholera']

X_all = np.zeros((T, N_NODES, D), dtype=np.float32)
for _, row in cases.iterrows():
    t   = week_to_t.get(row['week_start_date'])
    n   = int(row['node_idx'])
    d   = disease_names.index(row['disease']) if row['disease'] in disease_names else -1
    if t is not None and d >= 0:
        X_all[t, n, d] = np.log1p(float(row['cases']))

# Normalize each disease channel
scaler = StandardScaler()
X_flat = X_all.reshape(-1, D)
X_flat = scaler.fit_transform(X_flat)
X_all  = X_flat.reshape(T, N_NODES, D)
print(f'Node features: {X_all.shape}  (T x N x D = {T} x {N_NODES} x {D})')

In [ ]:
# ── ST-GNN Architecture: GraphSAGE + GRU ─────────────────────────────────────
SEQ_LEN    = 24   # 24-week encoder
PRED_LEN   = 4    # 4-week forecast
HIDDEN     = 64
GNN_LAYERS = 2
EPOCHS     = 50
LR         = 1e-3
BATCH_WEEKS= 8    # number of time steps per batch

class STGNN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim=D, n_layers=2):
        super().__init__()
        self.convs = nn.ModuleList([SAGEConv(in_dim if i==0 else hidden, hidden)
                                    for i in range(n_layers)])
        self.gru   = nn.GRU(hidden, hidden, num_layers=2, batch_first=True, dropout=0.1)
        self.head  = nn.Linear(hidden, out_dim * PRED_LEN)
        self.relu  = nn.ReLU()

    def forward(self, x_seq, edge_index):
        # x_seq: [T, N, D]
        T_steps, N, _ = x_seq.shape
        spatial_out = []
        for t in range(T_steps):
            h = x_seq[t]  # [N, D]
            for conv in self.convs:
                h = self.relu(conv(h, edge_index))
            spatial_out.append(h)  # [N, hidden]
        # Stack: [N, T, hidden] -> GRU
        spatial_seq = torch.stack(spatial_out, dim=1)  # [N, T, hidden]
        gru_out, _ = self.gru(spatial_seq)
        last_hidden = gru_out[:, -1, :]  # [N, hidden]
        pred = self.head(last_hidden)    # [N, D*PRED_LEN]
        return pred.view(N, D, PRED_LEN)

model     = STGNN(D, HIDDEN).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = nn.HuberLoss()
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
X_tensor = torch.FloatTensor(X_all).to(DEVICE)

cutoff = int(T * 0.85)
train_end = cutoff
train_losses = []; val_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0; n_batches = 0
    for start in range(0, train_end - SEQ_LEN - PRED_LEN, BATCH_WEEKS):
        end = min(start + SEQ_LEN, train_end - PRED_LEN)
        x_in  = X_tensor[start:end]
        y_out = X_tensor[end:end+PRED_LEN].permute(1, 2, 0)  # [N, D, PRED_LEN]
        if x_in.shape[0] < SEQ_LEN // 2: continue
        optimizer.zero_grad()
        pred = model(x_in, edge_index)
        loss = criterion(pred, y_out)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item(); n_batches += 1
    avg_loss = epoch_loss / max(n_batches, 1)
    train_losses.append(avg_loss)

    # Validation
    model.eval()
    with torch.no_grad():
        x_val = X_tensor[cutoff-SEQ_LEN:cutoff]
        y_val = X_tensor[cutoff:cutoff+PRED_LEN].permute(1, 2, 0)
        val_pred = model(x_val, edge_index)
        val_loss = criterion(val_pred, y_val).item()
        val_losses.append(val_loss)
        scheduler.step(val_loss)

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Train: {avg_loss:.4f} | Val: {val_loss:.4f}')

print('Training complete.')

In [ ]:
# ── Metrics ──────────────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    x_test = X_tensor[cutoff-SEQ_LEN:cutoff]
    y_true_t = X_tensor[cutoff:cutoff+PRED_LEN].permute(1, 2, 0).cpu().numpy()
    y_pred_t = model(x_test, edge_index).cpu().numpy()

y_true_f = y_true_t.flatten()
y_pred_f = y_pred_t.flatten()
mae  = mean_absolute_error(y_true_f, y_pred_f)
rmse = np.sqrt(mean_squared_error(y_true_f, y_pred_f))
r2   = r2_score(y_true_f, y_pred_f)
print('=' * 45)
print('  ST-GNN VALIDATION METRICS')
print('=' * 45)
print(f'  MAE  (normalized log scale): {mae:.4f}')
print(f'  RMSE (normalized log scale): {rmse:.4f}')
print(f'  R2 score:                    {r2:.4f}')
print('=' * 45)

# Per-disease metrics
for di, disease in enumerate(disease_names):
    yt = y_true_t[:, di, :].flatten()
    yp = y_pred_t[:, di, :].flatten()
    mae_d = mean_absolute_error(yt, yp)
    r2_d  = r2_score(yt, yp)
    print(f'  {disease:<10}: MAE={mae_d:.4f} | R2={r2_d:.4f}')

In [ ]:
# ── Loss curves + forecast plots ─────────────────────────────────────────────
import matplotlib; matplotlib.use('Agg')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='#2563EB', linewidth=2)
axes[0].plot(val_losses,   label='Val Loss',   color='#DC2626', linewidth=2, linestyle='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Huber Loss')
axes[0].set_title('ST-GNN Training Loss Curve')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Forecast vs actual for top-burden district (by total malaria cases)
mal_by_node = cases[cases['disease']=='malaria'].groupby('node_idx')['cases'].sum()
top_node = int(mal_by_node.idxmax()) if len(mal_by_node) > 0 else 0
top_name = node_list[top_node]['name'] if top_node < len(node_list) else 'Unknown'

true_seq = scaler.inverse_transform(X_all[:, top_node, :])[:, 0]  # malaria channel
axes[1].plot(true_seq[-52:], label='Actual (last 52w)', color='#2563EB')
t_pred_arr = y_true_t[top_node, 0, :]  # actual in test period
p_pred_arr = y_pred_t[top_node, 0, :]  # predicted
offset = len(true_seq[-52:])
x_pred = range(offset - PRED_LEN, offset - PRED_LEN + PRED_LEN)
axes[1].plot(list(x_pred), [scaler.inverse_transform([[v,0,0]])[0,0] for v in t_pred_arr],
             'b--', label='Actual (forecast period)')
axes[1].plot(list(x_pred), [scaler.inverse_transform([[v,0,0]])[0,0] for v in p_pred_arr],
             'r-', linewidth=2, label='ST-GNN Forecast')
axes[1].set_title(f'Malaria Forecast: {top_name}')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle(f'ST-GNN Results  |  Val R2={r2:.3f}', fontweight='bold')
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'stgnn_results.png', dpi=150)
plt.show()
print('Plots saved.')

In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────────
import torch; from datetime import datetime
ckpt_path = DRIVE_OUTPUT + 'stgnn_best.pt'
torch.save({'state_dict': model.state_dict(), 'scaler': scaler,
            'node_list': node_list, 'disease_names': disease_names,
            'edge_index': edge_index.cpu(), 'SEQ_LEN': SEQ_LEN,
            'PRED_LEN': PRED_LEN, 'HIDDEN': HIDDEN,
            'metrics': {'MAE': float(mae), 'RMSE': float(rmse), 'R2': float(r2)},
            'trained_at': datetime.now().isoformat()}, ckpt_path)
from google.colab import files
files.download(ckpt_path)
print(f'Saved: {ckpt_path}')
print(f'MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f}')